# Forecast — corrida mensual

Este es el nodo que se programa **el día 1 de cada mes**.

Lee la fuente (que ya incluye el mes que cerró) y la salida de la corrida anterior. El mes recién
cerrado ya tenía forecast guardado, así que pasa a ser un punto de validación **sin recalcular
nada**: sólo se recalcula qué modelo es el mejor para cada serie y se proyectan los `horizon`
meses nuevos. Por eso esta corrida cuesta una fracción de la inicial.

Toda la configuración está en **`fc_oracle.py`**.

### Cómo queda el nodo en el pipeline (Elyra / OpenShift AI)

| Propiedad del nodo | Valor |
|---|---|
| **File Dependencies** | `fc_oracle.py`, `forecast_engine.py` |
| **Environment Variables** | origen: `ORA_USER`, `ORA_PASSWORD`, `ORA_DSN` · destino: `ORA_DEST_USER`, `ORA_DEST_PASSWORD`, `ORA_DEST_DSN` · opcionales: `FC_CUTOFF`, `FC_REESCRIBIR_DESDE`, `FC_DRY_RUN`, `FC_MAX_WORKERS` |
| **Runtime Image** | una con `pandas statsmodels joblib psutil oracledb prophet` |
| **CPU / RAM** | lo que le asignes: el motor lee el límite del pod (cgroups) y ajusta la cantidad de procesos solo |

La contraseña va como *secret* montado en variable de entorno, nunca en el notebook.
Si una celda lanza excepción, el nodo queda en **failed** — que es lo que querés que pase.

In [ ]:
# Celda de parámetros (tag `parameters`): Elyra los pasa por variable de entorno,
# papermill puede inyectarlos acá directamente.
import os

CUTOFF = os.getenv("FC_CUTOFF") or None        # "2026-07-01" para fijar el último mes cerrado
DRY_RUN = os.getenv("FC_DRY_RUN", "0") == "1"  # 1 = calcula y no escribe

# Desde dónde reescribir la tabla de salida:
#   "auto"  -> desde el corte (borra las proyecciones viejas, deja la historia)
#   "todo"  -> borra todo y reinserta
#   6       -> 6 meses antes del corte
#   fecha   -> "2025-01-01"
REESCRIBIR_DESDE = os.getenv("FC_REESCRIBIR_DESDE") or "auto"

print(f"CUTOFF={CUTOFF!r}  REESCRIBIR_DESDE={REESCRIBIR_DESDE!r}  DRY_RUN={DRY_RUN}")

## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())   # fc_oracle.py y forecast_engine.py viajan como File Dependencies

import pandas as pd
import fc_oracle as io
from forecast_engine import (PanelForecaster, available_models,
                             available_cpus, available_memory_gb, container_cpu_limit)

log = io.configurar_logging("forecast-mensual")
cfg = io.build_config(cutoff=CUTOFF)

log.info("corte=%s | horizonte=%d | backtest=%d | refit_step=%d",
         cfg.resolved_cutoff().date(), cfg.horizon, cfg.backtest_horizon, cfg.refit_step)
log.info("recursos: %d cpus usables (límite del pod: %s) | %.1f GB de RAM disponibles",
         available_cpus(), container_cpu_limit() or "sin límite", available_memory_gb() or -1)
log.info("modelos: %s", ", ".join(available_models(cfg.models)))
t_inicio = time.time()

## 2. Leer la fuente y la salida anterior

La entrada de este nodo son **las dos**: la fuente con la métrica real, y la tabla ancha de la
corrida anterior con una columna por modelo. Esa tabla es la que aporta el forecast que se había
hecho para el mes que ahora cerró.

Si no hay salida anterior (primera vez que corre este nodo, o se truncó la tabla), avisa y hace
el backtest completo en vez de fallar: tarda como la corrida inicial pero el pipeline no se corta.

In [ ]:
with io.conexion_origen() as conn:
    src = io.leer_fuente(conn, cfg)
if src.empty:
    raise RuntimeError("La fuente no devolvió filas: revisá SQL_FUENTE en fc_oracle.py")

with io.conexion_destino() as conn:
    io.validar_tabla(conn, cfg)
    previo = io.leer_salida_anterior(conn, cfg)

if previo is None:
    log.warning("No hay salida anterior en %s: esta corrida rehace el backtest completo",
                io.TABLA_SALIDA)
    if str(REESCRIBIR_DESDE).lower() == "auto":
        REESCRIBIR_DESDE = "todo"   # la tabla está vacía: hay que escribir también la validación
        log.warning("REESCRIBIR_DESDE pasa a 'todo'")

## 3. Correr el forecast

En el log mirá la línea `Backtest: N de M (serie, período) ya venían resueltos`: si `faltan 0`,
se reutilizó todo el histórico y sólo se calculó la proyección nueva.

In [ ]:
fc = PanelForecaster(cfg)
out = fc.run(src, previous=previo)

## 4. Control de la corrida

La elección del modelo usa una **ventana móvil de `score_window` meses** (12 por defecto): cada
primero de mes entra el que acaba de cerrar y sale el más viejo. Con corte julio se evalúa
ago-2025..jul-2026; el mes que viene será sep-2025..ago-2026.

Sale de la **evaluación**, no de la tabla: la fila vieja se queda guardada con el `Y_REAL` y los
forecasts de cada modelo que tenía, y el `DELETE` nunca la alcanza (arranca en el corte). Buscá en
el log la línea `selección: ventana móvil ...` para ver el rango y cuántos puntos se evaluaron por
serie — si dice menos de 12 es una serie nueva, sin historia suficiente todavía.

In [ ]:
io.resumen(fc, out, cfg)
fc.summary()

In [ ]:
# El mes que acaba de cerrar: forecast heredado de la corrida anterior + su valor real
mes = cfg.resolved_cutoff()
out[out[cfg.date_col] == mes].head(10)

In [ ]:
# Vista rápida para dejar evidencia en el notebook ejecutado que archiva el pipeline
serie = out[cfg.category_cols[0]].iloc[0]
columnas = ([cfg.date_col, cfg.actual_col]
            + sorted(c for c in out.columns if c.startswith(cfg.model_col_prefix))
            + [cfg.best_model_col, cfg.forecast_col, cfg.future_flag_col])
out.loc[out[cfg.category_cols[0]] == serie, columnas]

## 5. Guardar

`DELETE` del rango + `INSERT`, en una sola transacción: si el insert falla, el borrado se deshace
y la tabla queda como estaba.

Con `REESCRIBIR_DESDE="auto"` el rango arranca en el **corte**. Ejemplo con corte julio y
`horizon=12`:

```
mes:        ... may  jun  jul   ago  sep  oct ...  jul+1año
corrida     ─────────────┤                                    intacto (no se toca)
anterior                 │ borra ──────────────────────────┤  proyecciones viejas de jun
esta        ─────────────┤ inserta ────────────────────────┤  jul (ya con real) + 12 meses
```

Los meses previos al corte no se reescriben: quedan con el `BEST_MODEL` y el `YHAT` que se
decidieron en la corrida que los escribió, que es el registro fiel de qué se proyectó en su
momento. Si querés que toda la historia refleje la elección de hoy, corré con
`FC_REESCRIBIR_DESDE=todo` (o `12`, o una fecha).

In [ ]:
if DRY_RUN:
    log.warning("DRY_RUN activo: no se escribe nada en %s", io.TABLA_SALIDA)
else:
    with io.conexion_destino() as conn:
        io.guardar(conn, out, cfg, reescribir_desde=REESCRIBIR_DESDE)

log.info("corrida mensual OK en %.1fs", time.time() - t_inicio)